# 検証：Zero-Shot Hard Routing (メタデータによる強制割り当て)

## 概要
このノートブックでは、`Router` の `allowed_mask` 機能を活用し、入力データのメタデータに基づいて特定のシナプス（例：電卓シナプス）へのトラフィックを100%に強制する「Zero-Shot ルーティング」の仕組みを検証します。
ルーターに学習させることなく、特定のトークンを確実に対応するシナプスへ送ることができるかを確認します。


In [ ]:
import os
import sys

# Colab環境での実行用セットアップ
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sra_reference import SRAModel, CalculatorSynapse
from constants import PAD

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")


In [ ]:
# 1. モデルの初期化
VOCAB_SIZE = 128
DIM = 64
LAYERS = 2
NUM_SYNAPSES = 4
K = 2
SYN_HIDDEN = 128

model = SRAModel(
    vocab_size=VOCAB_SIZE, 
    dim=DIM, 
    layers=LAYERS, 
    num_synapses=NUM_SYNAPSES, 
    k=K, 
    syn_hidden=SYN_HIDDEN
).to(device)

print(f"初期のシナプス数: {model.blocks[0].router.num_synapses}")


In [ ]:
# 2. CalculatorSynapseの追加
# 確定的な計算を行うシナプスを追加します
def calc_factory():
    return CalculatorSynapse(dim=DIM)

def emb_factory():
    # ランダムな初期Embeddingを持つ
    return torch.randn(DIM)

model.add_custom_synapse(calc_factory, emb_factory)
model = model.to(device)

calc_synapse_idx = NUM_SYNAPSES # 追加されたシナプスのインデックス (元の数と等しい)
print(f"追加後のシナプス数: {model.blocks[0].router.num_synapses}")
print(f"CalculatorSynapse のインデックス: {calc_synapse_idx}")


In [ ]:
# 3. ダミーデータの作成
# B=1, T=10 の入力シーケンスを想定
BATCH_SIZE = 1
SEQ_LEN = 10
x = torch.randint(1, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN)).to(device)
y_in = torch.randint(1, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN)).to(device)

print(f"入力シーケンス (x): {x.shape}")


In [ ]:
# 4. Zero-Shot ルーティング (allowed_mask なしの場合の挙動)
model.eval()
with torch.no_grad():
    _, router_logits_normal, _ = model(x, y_in)
    
# 最後のブロックのルーティング重みを確認
last_block_logits = router_logits_normal[-1] # (B, T_total, num_synapses)
# target positions のみ取得 (入力後のT個)
last_block_logits_tgt = last_block_logits[:, x.size(1):, :]
weights_normal = torch.softmax(last_block_logits_tgt, dim=-1)

print("allowed_mask なしの場合のルーティング確率 (最初のトークン):")
for i in range(model.blocks[0].router.num_synapses):
    print(f"  Synapse {i}: {weights_normal[0, 0, i].item():.4f}")


In [ ]:
# 5. allowed_mask を用いた強制ルーティング
# 最初の3つのトークンだけ、CalculatorSynapseへ強制的にルーティングさせる
total_seq_len = x.size(1) + y_in.size(1)
allowed_mask = torch.ones((BATCH_SIZE, total_seq_len, model.blocks[0].router.num_synapses), dtype=torch.bool).to(device)

# target positions の最初の3トークンを計算タスクと判定した（メタデータ）と仮定
# T_target のインデックスは x.size(1) から始まる
target_start = x.size(1)

# 最初は全シナプス許可されているが、指定トークンだけは CalculatorSynapse 以外を False にする
for t in range(3):
    idx = target_start + t
    allowed_mask[0, idx, :] = False
    allowed_mask[0, idx, calc_synapse_idx] = True

# 再度 forward を実行。今回は SRAModel を改修せずにブロック単位で直接検証するか、
# SRABlock の forward に allowed_mask を渡せるか確認する必要があります。
# 実装を見ると、Router.forward には allowed_mask がありますが、SRAModel.forward には渡されていません。
# そのため、各ブロックに対して手動で処理を行うか、一時的に SRABlock を直接呼び出して検証します。

print("SRAModel に allowed_mask を渡す機能がないため、SRABlock 単体で検証を行います。")


In [ ]:
# 6. SRABlock 単体での検証
block = model.blocks[0]
h = torch.randn(BATCH_SIZE, SEQ_LEN, DIM).to(device)

# マスク作成 (B, T, NumSynapses)
# 最初の3トークンだけ CalculatorSynapse を強制
mask = torch.ones((BATCH_SIZE, SEQ_LEN, block.router.num_synapses), dtype=torch.bool).to(device)
for t in range(3):
    mask[0, t, :] = False
    mask[0, t, calc_synapse_idx] = True

with torch.no_grad():
    # _norm
    h_norm = block.norm(h)
    
    # 1. Mask なしのルーティング
    idx_nomask, weights_nomask, logits_nomask = block.router(h_norm)
    
    # 2. Mask ありのルーティング
    idx_mask, weights_mask, logits_mask = block.router(h_norm, allowed_mask=mask)

print("--- Mask なし ---")
print("最初のトークンの選択インデックス:", idx_nomask[0, 0].tolist())
print("最初のトークンのルーティング確率:")
for i in range(block.router.num_synapses):
    print(f"  Synapse {i}: {weights_nomask[0, 0, i].item():.4f}")

print("\n--- Mask あり (Calculator強制) ---")
print("最初のトークンの選択インデックス:", idx_mask[0, 0].tolist())
print("最初のトークンのルーティング確率:")
for i in range(block.router.num_synapses):
    print(f"  Synapse {i}: {weights_mask[0, 0, i].item():.4f}")

# 4番目のトークン（強制されていない）の確認
print("\n--- Mask あり (4番目のトークン、強制なし) ---")
print("選択インデックス:", idx_mask[0, 3].tolist())
print("ルーティング確率:")
for i in range(block.router.num_synapses):
    print(f"  Synapse {i}: {weights_mask[0, 3, i].item():.4f}")


## 考察
検証結果から、`allowed_mask` を使用することで、事前のファインチューニングや学習を一切行うことなく、
100%の確率で指定したシナプス（今回は CalculatorSynapse）にトラフィックを流せることが確認できました。

これにより、外部の分類器や正規表現、プロンプトのメタデータを用いて、モデルに確実な計算や事実検索（VectorDB）を強制する「Zero-Shot Hard Routing」の実現性が証明されました。

今後の課題としては、この `allowed_mask` を `SRAModel.forward()` から各層のルーターへ透過的に渡せるようにインターフェースを拡張することが挙げられます。
